# 40 — Gold dimensions
Deterministic surrogate keys (`xxhash64` of the business key) so dimensions can be rebuilt without
breaking fact foreign keys. Each dimension has an `-1` UNKNOWN member for unmatched facts.

In [ ]:
%run ./_common

In [ ]:
def save(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("gold", name))
    print(name, spark.table(fq("gold", name)).count())


# dim_date 2024-2027
dates = (spark.sql("SELECT explode(sequence(to_date('2024-01-01'), to_date('2027-12-31'), interval 1 day)) AS date")
         .select(F.date_format("date", "yyyyMMdd").cast("int").alias("date_sk"), "date",
                 F.year("date").alias("year"), F.quarter("date").alias("quarter"), F.month("date").alias("month"),
                 F.date_format("date", "MMMM").alias("month_name"), F.dayofweek("date").alias("day_of_week"),
                 F.date_format("date", "EEEE").alias("day_name"),
                 F.dayofweek("date").isin(1, 7).alias("is_weekend")))
save(dates, "dim_date")

# dim_currency
fx = spark.table(fq("silver", "currency_fx"))
cur = fx.select(sk("currency_code").alias("currency_sk"), "currency_code", "currency_name",
                F.col("rate_to_usd").cast("decimal(18,8)"), "rate_date")
unknown_cur = spark.createDataFrame([(-1, "UNK", "Unknown", None, None)],
                                    "currency_sk long, currency_code string, currency_name string, rate_to_usd decimal(18,8), rate_date date")
save(cur.unionByName(unknown_cur), "dim_currency")

# dim_bank
bank = spark.table(fq("silver", "bank")).select(sk("bank_id").alias("bank_sk"), "bank_id", "bic", "bank_name", "country")
unknown_bank = spark.createDataFrame([(-1, -1, "UNKNOWN", "Unknown", "ZZ")],
                                     "bank_sk long, bank_id int, bic string, bank_name string, country string")
save(bank.unionByName(unknown_bank), "dim_bank")

# dim_lc (current attributes; amount changes live in the fact via amendments)
lc = spark.table(fq("silver", "letter_of_credit")).select(
    sk("lc_id").alias("lc_sk"), "lc_id", "lc_number", "payment_terms", "status", "issue_date", "expiry_date")
save(lc, "dim_lc")

# dim_counterparty (SCD2)
hist = spark.table(fq("silver", "party_history")).select(
    sk("party_id", "eff_start_date").alias("counterparty_sk"),
    "party_id", F.col("party_name").alias("counterparty_name"), "country", "party_type", "risk_rating",
    "eff_start_date", "eff_end_date", "is_current")
unknown_cp = spark.createDataFrame(
    [(-1, -1, "UNKNOWN", "ZZ", "UNKNOWN", "NR", "1900-01-01", "9999-12-31", True)],
    "counterparty_sk long, party_id int, counterparty_name string, country string, party_type string, risk_rating string, "
    "eff_start_date string, eff_end_date string, is_current boolean"
).withColumn("eff_start_date", F.col("eff_start_date").cast("date")).withColumn("eff_end_date", F.col("eff_end_date").cast("date"))
save(hist.unionByName(unknown_cp), "dim_counterparty")